# Clase 7 · Notebook 1: construimos el pipeline paso a paso, con 1 000 viajes

**El problema.** El pipeline que corre en la nube es una cadena de funciones: traer el archivo, leerlo, guardarlo en bronze, limpiarlo en silver, resumirlo en gold y cargarlo a Snowflake. Si las dejas correr solas sin haberlas entendido, el día que algo falle no sabrás por dónde empezar. Aquí las construimos una por una, con **1 000 viajes** en lugar de 20 millones, y las probamos varias veces hasta sentirnos seguros.

**Cómo se usa.** Este notebook lo conduce el instructor, y tú lo sigues celda por celda en tu propio Code Editor. Cada celda de código empieza con un comentario `# Celda N`: es su número, y la clase lo usa para decir dónde estamos. Ejecuta una celda a la vez con **Shift + Enter** y espera a que termine antes de pasar a la siguiente.

**Qué escribe y dónde.** Todo lo que corres desde el Code Editor es **dev**, y no se puede cambiar: el código se niega a correr prod fuera de la máquina desplegada por el CI/CD. Este notebook corre contra el **dev de verdad**, sin laboratorio ni simulacros: Spark en el Code Editor lee y escribe en `s3://<tu-lake>/dev/...`, y la tabla de control es la real, `BIGDATA.CONTROL_DEV.WATERMARK` (la conexión sale del secreto `bigdata/snowflake`, con el rol `BIGDATA_DEV`). **Nada se escribe en el disco del Code Editor.** Lo que se escribe aquí es de verdad: por eso, al terminar, volvemos a correr `resetear_entorno` en dev para dejar todo limpio antes de pasar las funciones a los `.py`.

- **escribe** en S3, en `dev/` (el crudo y las capas de bronze), y **anota** en `BIGDATA.CONTROL_DEV.WATERMARK`;
- **crea** una tabla vacía en `BIGDATA.DEV` (Snowflake);
- **no toca** `prod/`, `BIGDATA.PROD` ni `CONTROL_PROD`.

**El límite.** `limite="1000"` recorta lo que se lee de cada fuente y **se aplica solo en bronze**, justo después de leer el crudo y antes de transformar o escribir. Silver y gold leen todo lo que hay en bronze.

**Qué importamos y qué escribimos.** Las funciones de la caja de herramientas de la casa (`traer_crudo_viajes`, `leer_crudo`, `registrar_control`, `cargar_gold`, entre otras) se **importan**, nunca se copian. Lo que construimos aquí son las funciones del pipeline, con **exactamente** las mismas firmas que tienen en `bronze.py`, `silver.py` y `gold.py`. Este notebook es **tu banco de trabajo**: aquí cambias o agregas pasos a cada función, los pruebas con `limite="1000"` contra el dev de verdad y, cuando funcionan, los copias al `.py` (la parte 10 te dice cuáles cambiaste).

| Parte | Qué construimos | Qué probamos |
|---|---|---|
| 0 | La preparación: kernel, repositorio, corrida y Spark | Que estamos en dev y que Spark enciende |
| 1 | El crudo | Que `leer_crudo` con límite devuelve siempre las mismas filas |
| 2 a 5 | La puerta de la tabla de control: `debe_ingestar` y `registrar_ingesta` | Las tres reglas, con meses de ensayo en la tabla de control real |
| 6 | Bronze con nuestras funciones: dos corridas | `REGISTRÓ` y después `SE SALTÓ` |
| 7 | Silver: limpiar y ver qué pasa si un mes entra dos veces | El problema que resuelve `deduplicar` |
| 8 | Gold: una pregunta de la casa | Nuestra función contra la de la casa |
| 9 | Snowflake: el `CREATE TABLE` a partir de los tipos de gold | Que la tabla nace con las columnas correctas |
| 10 | Del notebook al `.py` | Qué funciones cambiaste frente al repositorio y las pruebas del equipo |

> **Antes de empezar.** En esta máquina solo cabe un proceso de Spark a la vez. Si Airflow del Code Editor está encendido, apágalo con `cd ~/bigdata && make airflow-local-down`, y si dejaste abierto otro notebook con Spark encendido, ciérralo primero. Este notebook usa los `.jar` de S3A que bajó `make jars-s3a` (si no los tienes, córrelo una vez). La última celda de este notebook apaga Spark.

## 0. Preparación: el kernel, el repositorio y la corrida

### 0.1 El kernel y la raíz del repositorio

El **kernel** es el programa Python que ejecuta las celdas. Elige siempre **Python (bigdata)**, que es el entorno `.venv` del proyecto. No elijas los kernels llamados *Glue PySpark* o *SparkMagic*: abren una sesión de AWS Glue en la nube y esa sesión cobra mientras está abierta. La celda imprime qué Python usas y deja el repositorio en la ruta de búsqueda, para poder escribir `from pipeline... import ...`.

In [ ]:
# Celda 1
import os, shutil, sys
from pathlib import Path

# Java: el kernel del notebook no lee ~/.bashrc, así que puede no ver el Java que instaló make setup.
if shutil.which("java") is None:
    for candidato in [os.environ.get("JAVA_HOME"), str(Path.home() / ".local/opt/java17")]:
        if candidato and (Path(candidato) / "bin/java").exists():
            os.environ["JAVA_HOME"] = candidato
            os.environ["PATH"] = f"{candidato}/bin:" + os.environ["PATH"]
            break

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este kernel no es el del proyecto. Elige «Python (bigdata)» arriba a la derecha y reinicia el notebook.")

# La raíz del repositorio es la carpeta que contiene pyproject.toml.
RAIZ = Path.home() / "bigdata"
for carpeta in [Path.cwd(), *Path.cwd().parents]:
    if (carpeta / "pyproject.toml").exists():
        RAIZ = carpeta
        break
sys.path.insert(0, str(RAIZ / "src"))   # así se puede hacer «from pipeline... import ...»
print("Repositorio:", RAIZ)

### 0.2 La corrida: lo que decide cada llamada

Antes de ejecutar un solo paso hay que saber con qué reglas corre. Esas reglas viven en un objeto, la **corrida**, que arma `cargar_configuracion`. Sale de tres lugares, en este orden de prioridad: lo que pasas como argumento, lo que dice `pipeline.yaml` (en la raíz del repositorio) y los valores por defecto. El **entorno** no se elige: desde aquí siempre es `dev`.

Aquí pedimos `limite="1000"`: cada fuente se lee recortada a 1 000 filas, siempre las mismas. La función `banner` imprime el encabezado que van a mostrar **todas** las corridas del curso (consola, notebook, Airflow, Glue y prod): entorno, cómputo, paso, mes, límite, qué lee, qué escribe y cuál tabla de control usa.

Importamos cada herramienta desde su archivo (por ejemplo `pipeline.herramientas.corrida`) para que siempre sepas de dónde viene; el código está en `src/pipeline/herramientas/<archivo>.py`.

In [ ]:
# Celda 2
from pipeline.herramientas.corrida import banner, cargar_configuracion
from pipeline.herramientas.rutas import donde_estoy

corrida = cargar_configuracion(limite="1000")
print(corrida)
print()
print(banner(corrida, "bronze", "2024-01"))

**Qué observar.** `entorno='dev'`, `computo='code_editor'` y `limite='1000'`. Las rutas de `Lee` y `Escribe` empiezan con `s3://<tu-bucket>/dev/`: es el dev de verdad. En `Lee` aparece la carpeta `_crudo/`, el archivo tal como se descargó; en `Escribe`, la carpeta de bronze. La última línea es la tabla de control que consulta y anota este paso: `BIGDATA.CONTROL_DEV.WATERMARK`.

La función `donde_estoy` imprime, en una tabla, todas las carpetas de un entorno. Es la respuesta rápida a «¿en qué carpeta quedó esto?».

In [ ]:
# Celda 3
donde_estoy("dev")

### 0.3 Spark con la misma configuración del pipeline

Los cómputos del pipeline son cuatro: `code_editor`, `glue_small`, `glue_medium` y `glue_large`. En este notebook usamos el primero: Spark en esta máquina (`local[2]`, 2 GB de memoria). El mismo Spark lee y escribe en S3 con dos `.jar` que bajó `make jars-s3a`.

In [ ]:
# Celda 4
from pipeline.engine import get_spark

spark = get_spark("code_editor")
print("Versión de Spark:", spark.version)
print("Master          :", spark.sparkContext.master)

## 1. El crudo: traer el archivo y leerlo con un límite

**El problema.** Bronze guarda lo que llega de la fuente, pero antes hay que tener el archivo en tu lake. Esa es la tarea de `traer_crudo_viajes`, `traer_crudo_taxi_zones` y `traer_crudo_clima`: si el archivo ya está en `_crudo/`, no se vuelve a descargar; si está en la ruta del diseño anterior, se mueve; y si no, se descarga desde la fuente. Cada función devuelve la ruta completa en S3 y dice en voz alta qué hace.

La primera vez, el archivo de viajes de enero pesa unos 450 MB, se descarga y se sube a `s3://<tu-lake>/dev/bronze/_crudo/`, y la celda tarda unos minutos. Las siguientes veces el archivo ya está en S3 y termina en un segundo.

In [ ]:
# Celda 5
from pipeline.herramientas.crudo import (
    traer_crudo_clima,
    traer_crudo_taxi_zones,
    traer_crudo_viajes,
)

ruta_viajes = traer_crudo_viajes(corrida.entorno, "2024-01")
ruta_zonas = traer_crudo_taxi_zones(corrida.entorno)
ruta_clima = traer_crudo_clima(corrida.entorno, "2024-01")
print()
print("Viajes:", ruta_viajes)
print("Zonas :", ruta_zonas)
print("Clima :", ruta_clima)

### 1.1 Leer el crudo con un límite

`leer_crudo(spark, entorno, tabla, mes, limite)` devuelve un DataFrame. Los viajes vienen en Parquet; las zonas, en un CSV leído todo como texto; el clima, en un JSON que la función ya convierte a una fila por día. `limite` es el número de filas que se leen (`corrida.limite_filas` vale `1000` en este notebook) o `None` para leer todo.

In [ ]:
# Celda 6
from pipeline.herramientas.capas import leer_crudo

viajes = leer_crudo(spark, "dev", "fhvhv", "2024-01", limite=corrida.limite_filas)
print("Filas leídas:", viajes.count())
print("Columnas    :", len(viajes.columns))
viajes.select("hvfhs_license_num", "pickup_datetime", "PULocationID", "trip_miles", "base_passenger_fare").show(5, truncate=False)

### 1.2 ¿Son siempre las mismas 1 000 filas?

Esta pregunta decide si más adelante podremos comparar corridas. Un `limit` suelto de Spark puede devolver filas distintas cada vez, según qué tarea termine primero. `leer_crudo` aplica el límite de forma **determinista** (`coalesce(1).limit(n)`): toma siempre las mismas primeras `n` filas del archivo. Lo comprobamos leyendo dos veces y comparando fila por fila.

In [ ]:
# Celda 7
otra_lectura = leer_crudo(spark, "dev", "fhvhv", "2024-01", limite=corrida.limite_filas)
print("¿Las dos lecturas son idénticas?", viajes.collect() == otra_lectura.collect())

Las zonas y el clima son fuentes pequeñas. Las leemos igual, para tener sus números a mano: 265 zonas y 31 días de enero.

In [ ]:
# Celda 8
zonas = leer_crudo(spark, "dev", "taxi_zones")
clima = leer_crudo(spark, "dev", "clima", "2024-01")
print("Zonas:", zonas.count(), "filas | columnas:", zonas.columns)
print("Clima:", clima.count(), "filas | columnas:", clima.columns)

## 2. La puerta de la tabla de control, primera versión

**El problema.** Cada ingesta de bronze escribe un lote **nuevo** y nunca borra los anteriores. Si corremos bronze dos veces para enero, enero queda dos veces. Hace falta que el pipeline pregunte antes de trabajar: «¿ya hice este mes?». Esa pregunta se contesta consultando la **tabla de control**: una tabla de Snowflake donde cada paso anota una fila cuando termina bien. Hay una por entorno, `BIGDATA.CONTROL_DEV.WATERMARK` y `BIGDATA.CONTROL_PROD.WATERMARK`, y la clave de consulta es `(TABLA, CAPA, MES)`.

La regla de la puerta es una sola, y la vamos a construir hasta que quede completa:

| ¿Hay registro? | ¿`reprocesar`? | Qué hace el pipeline |
|---|---|---|
| No | `false` | **Ingesta** |
| Sí | `false` | **Salta**, sea cual sea el número de filas, el cómputo o el límite del registro |
| (da igual) | `true` | **Ingesta siempre** (reprocesar es pedirlo a propósito) |

### 2.1 Mirar la tabla de verdad

`consultar_control(entorno, tabla, capa, mes)` devuelve la lista de registros de esa clave, del más antiguo al más reciente. Si la tabla no tiene nada para enero, devuelve una lista vacía. Esta consulta sí va a Snowflake, con el rol `BIGDATA_DEV`, y es solo de lectura.

In [ ]:
# Celda 9
from pipeline.herramientas.control import consultar_control, tabla_de_control

print("La tabla de control de dev es:", tabla_de_control("dev"))
registros = consultar_control("dev", "fhvhv", "bronze", "2024-01")
print("Registros de bronze para fhvhv, 2024-01:", registros)

Como acabamos de resetear dev, la lista sale vacía: ningún paso ha terminado todavía. Si sale un registro, es un resto de una corrida anterior: repite el reseteo.

### 2.2 Ensayar con meses de ensayo

Vamos a equivocarnos varias veces mientras construimos la puerta, y la tabla de control es la real. El riesgo: una fila de «ya procesado» para enero de 2024 sin datos detrás haría que el pipeline salte enero, que es justo el error que la tabla de control debe evitar. Por eso todos los ensayos de las partes 2 a 5 usan **meses de 2023**, que el pipeline de este curso nunca procesa: la clave `(TABLA, CAPA, MES)` es distinta y enero de 2024 queda intacto. Esas filas de ensayo se borran con el `resetear_entorno` del final.

In [ ]:
# Celda 10
MES_ENSAYO = "2023-12"   # un mes que el pipeline nunca procesa

print("Mes de ensayo:", MES_ENSAYO)
print("Registros de bronze para fhvhv, mes de ensayo:", consultar_control("dev", "fhvhv", "bronze", MES_ENSAYO))
print("Registros de bronze para fhvhv, 2024-01      :", consultar_control("dev", "fhvhv", "bronze", "2024-01"))

### 2.3 `debe_ingestar`, versión 1: ¿hay registro?

La primera versión solo responde a la columna «¿Hay registro?»: si no hay registro, se ingesta; si lo hay, se salta. Ya veremos qué le falta. La firma es la misma que tiene en `bronze.py`: `debe_ingestar(corrida, tabla, mes) -> bool`. Recibe la `corrida` (de ahí sale el entorno), la `tabla` (`fhvhv`, `taxi_zones` o `clima`) y el `mes`. La capa es siempre `"bronze"` porque esta es la puerta de bronze.

In [ ]:
# Celda 11
from pipeline.herramientas.corrida import Corrida

CAPA = "bronze"


def debe_ingestar(corrida: Corrida, tabla: str, mes: str) -> bool:
    registros = consultar_control(corrida.entorno, tabla, CAPA, mes)
    hay_registro = len(registros) > 0
    return not hay_registro


print("Sin registro, ¿ingestamos fhvhv del mes de ensayo?", debe_ingestar(corrida, "fhvhv", MES_ENSAYO))


Funciona para el primer caso: no hay registro, así que `debe_ingestar` dice `True`. Pero para probar el segundo caso necesitamos que exista un registro, y eso lo escribe otra función (en la tabla real, con el mes de ensayo). La construimos ahora.

## 3. `registrar_ingesta`: anotar solo cuando terminó bien

**El problema.** ¿Cuándo se anota que un mes ya se procesó? Si se anota **antes** de escribir y la escritura se cae a la mitad, la tabla dice «procesado» pero los datos no están: el mes nunca se vuelve a intentar, porque la puerta lo salta. Si se anota **después** y la escritura falla, no queda registro y el mes se puede reintentar. Por eso la regla es: **se registra al final y solo si todo salió bien**.

`registrar_control` es la función de la casa que escribe una fila en la tabla de control. Nuestra `registrar_ingesta` solo la llama con los datos de bronze: la tabla, la capa `"bronze"`, el mes, las filas **escritas**, el inicio, la fuente y las filas **leídas** (estas dos últimas son solo informativas). Su firma es la de `bronze.py`: `registrar_ingesta(corrida, tabla, mes, fuente, leidas, filas, inicio)`.

In [ ]:
# Celda 12
from datetime import UTC, datetime

from pipeline.herramientas.control import registrar_control
from pipeline.herramientas.corrida import Corrida

CAPA = "bronze"


def registrar_ingesta(
    corrida: Corrida,
    tabla: str,
    mes: str,
    fuente: str,
    leidas: int,
    filas: int,
    inicio: datetime,
) -> None:
    registrar_control(corrida, tabla, CAPA, mes, filas, inicio, fuente, leidas)


inicio = datetime.now(UTC).replace(microsecond=0)
registrar_ingesta(corrida, "fhvhv", MES_ENSAYO, "https://fuente/viajes", 1000, 1000, inicio)
for registro in consultar_control("dev", "fhvhv", "bronze", MES_ENSAYO):
    print(registro)


**Qué observar.** Ahora hay una fila con la tabla, la capa, el mes, `filas`, `leidas`, el `computo`, el `limite` y `reprocesar`. Las tres últimas columnas se guardan solo para informar; la puerta no las mira. Volvemos a preguntar a la puerta, que ahora sí tiene un registro:

In [ ]:
# Celda 13
print("Con registro, ¿ingestamos fhvhv del mes de ensayo?", debe_ingestar(corrida, "fhvhv", MES_ENSAYO))
print("¿Y el clima del mismo mes?                        ", debe_ingestar(corrida, "clima", MES_ENSAYO))
print("¿Y fhvhv de otro mes (2023-11)?                   ", debe_ingestar(corrida, "fhvhv", "2023-11"))

La puerta salta solo la clave `(fhvhv, bronze, 2023-12)`. El clima del mismo mes y los viajes de noviembre siguen abiertos.

### 3.1 Por qué se registra al final

Lo demostramos con una ingesta de mentira que puede fallar a la mitad. Usamos otro mes de ensayo (`2023-10`), que no tiene registros. Primero hacemos que la escritura se caiga; después, que termine bien.

In [ ]:
# Celda 14
MES_FALLO = "2023-10"


def ingesta_de_mentira(corrida, tabla, mes, falla):
    if not debe_ingestar(corrida, tabla, mes):
        return "SE SALTÓ"
    inicio = datetime.now(UTC).replace(microsecond=0)
    # Aquí irían traer el crudo, leerlo y escribir el lote.
    if falla:
        raise RuntimeError("se cayó la escritura a la mitad")
    registrar_ingesta(corrida, tabla, mes, "https://fuente/viajes", 1000, 1000, inicio)
    return "REGISTRÓ"


try:
    ingesta_de_mentira(corrida, "fhvhv", MES_FALLO, falla=True)
except RuntimeError as error:
    print("La ingesta falló:", error)
print("Registros después del fallo:", len(consultar_control("dev", "fhvhv", "bronze", MES_FALLO)))
print("Reintento:", ingesta_de_mentira(corrida, "fhvhv", MES_FALLO, falla=False))
print("Otro intento:", ingesta_de_mentira(corrida, "fhvhv", MES_FALLO, falla=False))

**Qué observar.** Tras el fallo no queda ningún registro, así que el mes se puede reintentar. El reintento termina bien y registra (`REGISTRÓ`). El siguiente intento encuentra el registro y no hace nada (`SE SALTÓ`). Si hubiéramos registrado al principio, el reintento habría quedado bloqueado por una fila que dice «procesado» sin que lo estuviera.

## 4. La puerta, versión final: `reprocesar`

**El problema.** Hay motivos legítimos para repetir un mes: la fuente corrigió sus datos, el cómputo se cambió, el equipo quiere medir otra cosa. La versión 1 no permite repetir nunca. La corrida trae una perilla para eso, `reprocesar`: si vale `true`, la puerta ignora la tabla de control y deja pasar. La herramienta de la casa `mes_ya_procesado(corrida, tabla, capa, mes)` ya contiene esa regla completa (la tabla de verdad de arriba) y responde `True` cuando hay que **saltar**. Nuestra puerta solo invierte la pregunta: si ya está registrado, no se ingesta; en cualquier otro caso, sí. Esta es la versión que tiene `bronze.py`. Reprocesar **no borra nada**: cada corrida deja su fila y la historia queda completa.

In [ ]:
# Celda 15
from pipeline.herramientas.control import mes_ya_procesado


def debe_ingestar(corrida: Corrida, tabla: str, mes: str) -> bool:
    ya_esta_registrado = mes_ya_procesado(corrida, tabla, CAPA, mes)
    if ya_esta_registrado:
        return False
    return True


## 5. Probar las tres reglas

Una función no está lista hasta que probamos sus tres casos. Armamos cada caso con su propio mes de ensayo, que no tiene registros (la tabla es la real y no se vacía): una corrida con `reprocesar` en el valor que toca (`dataclasses.replace` copia la corrida cambiando solo ese campo) y, en el segundo y tercer caso, un registro previo. Al lado ponemos lo que la tabla de verdad de la parte 2 dice que debe pasar.

In [ ]:
# Celda 16
from dataclasses import replace


def caso(mes, hay_registro, reprocesar):
    c = replace(corrida, reprocesar=reprocesar)
    if hay_registro:
        registrar_ingesta(c, "fhvhv", mes, "https://fuente/viajes", 1000, 1000, datetime.now(UTC))
    return debe_ingestar(c, "fhvhv", mes)


print(f"{'hay registro':<14}{'reprocesar':<12}{'mi puerta':<11}{'debe ser':<10}")
# (mes de ensayo, hay registro, reprocesar, qué debe pasar según la tabla de verdad)
casos = [("2023-07", False, False, True), ("2023-06", True, False, False), ("2023-05", True, True, True)]
for mes, hay_registro, reprocesar, esperado in casos:
    resultado = caso(mes, hay_registro, reprocesar)
    veredicto = "ingesta" if resultado else "salta"
    debe_ser = "ingesta" if esperado else "salta"
    print(f"{str(hay_registro):<14}{str(reprocesar):<12}{veredicto:<11}{debe_ser:<10}{'coincide' if resultado == esperado else 'DIFERENTE'}")


Las tres filas deben decir `coincide`: sin registro se ingesta, con registro se salta, y con `reprocesar` se ingesta siempre.

Con esto, nuestras dos funciones (`debe_ingestar` y `registrar_ingesta`) quedaron como las que trae `bronze.py`; la parte 10 lo comprueba. Comprobamos que los ensayos no tocaron enero de 2024 y seguimos con la tercera función de bronze.

In [ ]:
# Celda 17
print("Registros de ensayo (meses de 2023): se borrarán con resetear_entorno al final.")
for tabla_ in ["fhvhv", "taxi_zones", "clima"]:
    print(f"Registros de bronze para {tabla_}, 2024-01:", len(consultar_control("dev", tabla_, "bronze", "2024-01")))

## 6. Bronze con nuestras funciones: dos corridas con 1 000 viajes

Falta la tercera función de bronze, `agregar_metadatos(df, fuente, archivo_origen, id_lote, fecha_ingesta)`. Su tarea sería agregar a cada fila de qué lote, de qué archivo y de qué momento viene, más una huella de la fila (`_hash_fila`). **Hoy la dejamos como viene en el repositorio: no agrega nada.** Es un problema a propósito, y lo decimos en voz alta: bronze guarda las filas tal cual y nada dice de dónde vienen. Más adelante veremos qué le cuesta eso a silver. Escribir esta función es parte del trabajo de tu equipo.

La marca `# PENDIENTE(equipo)` dentro de la función es lo que buscan las pruebas del equipo para decir `PENDIENTE` en lugar de `FALLA`.

In [ ]:
# Celda 18
from pyspark.sql import DataFrame


def agregar_metadatos(
    df: DataFrame, fuente: str, archivo_origen: str, id_lote: str, fecha_ingesta: datetime
) -> DataFrame:
    # PENDIENTE(equipo): hoy esta función devuelve df tal cual, sin metadatos. Por eso bronze
    # no guarda de qué lote ni de qué archivo vino cada fila, y silver no puede deduplicar.
    return df


### 6.1 Poner nuestras funciones a trabajar en el pipeline

El pipeline ya trae su propio `main` de bronze, que hace la receta completa para las tres tablas (fhvhv, taxi_zones y clima): puerta, crudo, lectura, metadatos, escritura del lote y registro. Para que use **nuestras** tres funciones en este notebook, sin tocar todavía los archivos `.py`, usamos `usar_mis_funciones`. Funciona solo en este proceso: si reinicias el kernel, vuelven las del repositorio.

In [ ]:
# Celda 19
from pipeline.herramientas.equipo import usar_mis_funciones

usar_mis_funciones(
    agregar_metadatos=agregar_metadatos,
    debe_ingestar=debe_ingestar,
    registrar_ingesta=registrar_ingesta,
)

### 6.2 Primera corrida: `REGISTRÓ`

Esta corrida es de verdad: descarga lo que falte, escribe en `s3://<tu-lake>/dev/bronze/` y anota en `BIGDATA.CONTROL_DEV.WATERMARK`. `correr_bronze` es una de las cuatro funciones que corren los pasos: la misma que ejecuta `make correr PASO=bronze` y la que llama la primera tarea del DAG `pipeline_taxi`. Imprime el encabezado, hace el trabajo y cierra con una línea por tabla: cuántas filas leyó, cuántas escribió y si **se saltó** o **registró**. Con `limite="1000"` cada fuente se recorta a 1 000 filas (las zonas son 265 y el clima 31, así que entran completas).

In [ ]:
# Celda 20
from pipeline.correr import correr_bronze

resultados = correr_bronze(limite="1000")

**Qué observar.** El lote se escribió en S3, en `dev/bronze/`, y el registro está en `BIGDATA.CONTROL_DEV.WATERMARK`. En `Cierre` debe haber tres líneas con `REGISTRÓ en la tabla de control`: `fhvhv`, `taxi_zones` y `clima`. Las líneas `YA ESTÁ`, `LEE`, `ESCRIBE`, `SUBE`, `CONSULTA` y `REGISTRA` que aparecen antes dicen en voz alta cada ruta y cada tabla que se toca.

### 6.3 Segunda corrida, sin cambiar nada: `SE SALTÓ`

In [ ]:
# Celda 21
resultados = correr_bronze(limite="1000")

Ahora las tres líneas dicen `SE SALTÓ (ya registrado y reprocesar=false)`. La puerta consultó la tabla de control, encontró el registro de la primera corrida y no hizo nada: no hay lote nuevo, no hay filas repetidas. Mira lo que quedó anotado y las carpetas del lake:

In [ ]:
# Celda 22
for registro in consultar_control("dev", "fhvhv", "bronze", "2024-01"):
    print(registro)

In [ ]:
# Celda 23
from pipeline.herramientas.archivos import listar_subcarpetas
from pipeline.herramientas.rutas import ruta_lake

carpeta = ruta_lake("dev", "bronze", "fhvhv", "2024-01")
print(carpeta)
print(sorted(listar_subcarpetas(carpeta)))

**Qué observar.** Hay **una sola** carpeta `lote=<fecha>_<código>/`, a pesar de las dos corridas. Y el registro dice `filas=1000`, `leidas=1000`, `computo=code_editor` y `limite=1000`.

Reprocesar a propósito (`reprocesar=True`) crearía un segundo lote. **Todavía no lo hacemos**: más adelante lo haremos con la consola de Athena abierta, para ver con nuestros propios ojos qué le pasa a silver y a gold.

> **Para recordar.** La puerta funciona igual en todos lados: en un notebook, con `make`, en Airflow y en Glue. Aquí ya consultó la tabla de control real, `BIGDATA.CONTROL_DEV.WATERMARK`: con el mes registrado y `reprocesar` sin marcar, la respuesta es `SE SALTÓ`. Al final, `resetear_entorno` borra estos registros y vuelve a abrir la puerta.

## 7. Silver: limpiar, y ver qué pasa si un mes entra dos veces

**El problema.** Bronze guarda las filas como llegaron: viajes con distancia cero, duraciones imposibles, tipos mezclados. Silver es el **espejo limpio** de cada fuente: una tabla por fuente, sin join, con cada columna en su tipo y sin los viajes que no pueden ser reales.

Las funciones de limpieza son de la casa (`limpiar_viajes`, `clima_diario` y `limpiar_taxi_zones`, en `silver.py`). Aquí las probamos sobre el lote de bronze que acabamos de escribir. **No escribimos silver todavía**: solo miramos qué haría. Lo escribe el paso silver, que correremos con las tareas del DAG. Silver lee **todo** lo que hay en bronze: el límite ya se aplicó antes, al leer el crudo.

In [ ]:
# Celda 24
from pipeline import silver
from pipeline.herramientas.capas import leer_capa

bronze_viajes = leer_capa(spark, "dev", "bronze", "fhvhv", "2024-01")
silver_viajes = silver.limpiar_viajes(bronze_viajes, ["2024-01"])

print("Filas en bronze :", bronze_viajes.count())
print("Filas en silver :", silver_viajes.count())
print("Descartadas     :", bronze_viajes.count() - silver_viajes.count())
silver_viajes.select("pickup_datetime", "empresa", "distancia_mi", "duracion_min", "tarifa_base", "total_pagado").show(5, truncate=False)

**Qué observar.** Silver descarta los viajes que incumplen las reglas (distancia entre 0 y 100 millas, tarifa mayor que cero, duración entre 1 y 180 minutos) y deja cada columna con un nombre y un tipo: `distancia_mi` y `duracion_min` son números, `empresa` ya no es una licencia como `HV0003` sino `Uber`. `leer_capa` de bronze junta **todos** los lotes completos del mes.

Las otras dos fuentes pasan por sus propias funciones de limpieza: las zonas, con el identificador en entero, y el clima, con la fecha como fecha.

In [ ]:
# Celda 25
silver_zonas = silver.limpiar_taxi_zones(leer_capa(spark, "dev", "bronze", "taxi_zones"))
silver_clima = silver.clima_diario(leer_capa(spark, "dev", "bronze", "clima", "2024-01"))
print("Zonas en silver:", silver_zonas.count(), "| columnas:", silver_zonas.columns)
print("Clima en silver:", silver_clima.count(), "| columnas:", silver_clima.columns)

### 7.1 ¿Y si el mes entra dos veces?

Bronze solo agrega: si alguien reprocesa enero, bronze tiene **dos lotes** de enero con las mismas filas. Silver lee todos los lotes del mes. Lo simulamos sin escribir nada: unimos el lote consigo mismo, como si enero hubiera entrado dos veces, y lo pasamos por la misma limpieza.

In [ ]:
# Celda 26
dos_lotes = bronze_viajes.unionByName(bronze_viajes)
print("Filas de bronze si enero entró dos veces:", dos_lotes.count())
print("Filas de silver con esos dos lotes      :", silver.limpiar_viajes(dos_lotes, ["2024-01"]).count())
print("Filas de silver con un solo lote        :", silver_viajes.count())

**Qué observar.** Con dos lotes, silver tiene exactamente el doble de filas. Nadie recibió ningún error: cada viaje simplemente cuenta dos veces. Quien debe impedirlo es silver, con una función: `deduplicar(df)`. La dejamos **como viene en el repositorio: devuelve la tabla tal cual**, sin deduplicar, y lo decimos en voz alta: es el segundo problema a propósito. Su firma es la de `silver.py`: `deduplicar(df) -> DataFrame`. Lo comprobamos:

In [ ]:
# Celda 27
def deduplicar(df: DataFrame) -> DataFrame:
    # PENDIENTE(equipo): hoy esta función devuelve df tal cual. Por eso, al reprocesar un mes,
    # silver tiene cada fila dos veces (una por lote de bronze) y gold las multiplica.
    return df


solo_una_vez = deduplicar(silver.limpiar_viajes(dos_lotes, ["2024-01"]))
print("Filas después de deduplicar:", solo_una_vez.count(), "(debería ser", silver_viajes.count(), ")")


Quedan las mismas filas que antes de `deduplicar`: la función no hace nada. Su idea, escrita en una frase, es «dejar una sola fila por cada fila distinta», y para eso necesita saber qué filas son la misma. Bronze podría guardar una huella por fila (esa es la función `agregar_metadatos` que dejamos pendiente) y silver deduplicaría por esa huella. Las dos funciones van de la mano, y las dos son del equipo.

## 8. Gold: una pregunta de la casa, no la del Taller

**El problema.** Silver deja tablas limpias, pero nadie decide con una tabla de un millón de viajes. Gold responde **una pregunta de negocio** con una tabla pequeña. La pregunta de la casa, para aprender la receta, es:

> **¿A qué hora del día y en qué barrio se paga más por milla?**

No es la pregunta de ningún equipo del Taller 2: la tuya es otra y la escribes tú. Aquí solo construimos el ejemplo.

### 8.1 Juntar las tres tablas de silver

La función `enriquecer` (de la casa, en `silver.py`) une los viajes con las zonas (por `pu_location_id`, para saber el barrio de recogida) y con el clima (por la fecha). Son `left join`: no se pierde ningún viaje.

In [ ]:
# Celda 28
enriquecido = silver.enriquecer(silver_viajes, silver_zonas, silver_clima)
print("Viajes enriquecidos:", enriquecido.count(), "| columnas nuevas:", [c for c in enriquecido.columns if c in ("borough", "zona", "lluvia_mm", "temp_media_c")])
enriquecido.select("pickup_datetime", "borough", "zona", "distancia_mi", "total_pagado").show(5, truncate=False)

### 8.2 La franja del día

Para responder «a qué hora» hay que convertir la hora de recogida en una franja: madrugada (de 0 a 5 h), mañana (6 a 11), tarde (12 a 17) y noche (18 a 23). Probamos la idea sobre las horas del día antes de usarla.

In [ ]:
# Celda 29
from pyspark.sql import functions as F

horas = spark.range(0, 24).withColumnRenamed("id", "hora")
franja = (
    F.when(F.col("hora") < 6, "madrugada")
    .when(F.col("hora") < 12, "mañana")
    .when(F.col("hora") < 18, "tarde")
    .otherwise("noche")
)
horas.groupBy(franja.alias("franja")).agg(F.min("hora").alias("desde"), F.max("hora").alias("hasta")).orderBy("desde").show()

### 8.3 Agrupar y calcular la tarifa por milla

Agrupamos por fecha, barrio y franja. De cada grupo guardamos los viajes, la suma de las millas y la suma de lo pagado. La **tarifa por milla** es el cociente de las **sumas**, no el promedio de las tarifas de cada viaje: así un viaje de 0.1 millas no pesa igual que uno de 20. Redondeamos al final, para que el redondeo de las sumas no mueva el cociente. Si el barrio es nulo (una zona que no está en el lookup), lo llamamos `Desconocido` en lugar de perder el viaje.

Esta es la función completa, con su franja del día y sus listas. Su firma es la de `gold.py`: recibe el DataFrame enriquecido y devuelve la tabla de gold. Para agregar **tu** tabla, escribes otra función y la sumas a `TABLAS_GOLD` y a `LLAVES_DE_GOLD` (gold exige que esas columnas no se repitan ni sean nulas).

In [ ]:
# Celda 30
COLUMNAS_TARIFA_POR_MILLA = [
    "fecha",
    "borough",
    "franja",
    "viajes",
    "distancia_total_mi",
    "ingreso_total",
    "tarifa_por_milla",
]

# Para los viajes cuya zona de recogida no está en el lookup (el barrio queda nulo).
BARRIO_DESCONOCIDO = "Desconocido"


def _franja_del_dia(hora: F.Column) -> F.Column:
    """madrugada (0 a 5 h), mañana (6 a 11), tarde (12 a 17) o noche (18 a 23)."""
    return (
        F.when(hora < 6, "madrugada")
        .when(hora < 12, "mañana")
        .when(hora < 18, "tarde")
        .otherwise("noche")
    )


def tarifa_por_milla_por_barrio(enriquecido: DataFrame) -> DataFrame:
    """Por fecha, barrio y franja del día: viajes, millas, ingreso y cuánto se paga por milla."""
    con_franja = enriquecido.select(
        F.col("pickup_date").alias("fecha"),
        F.coalesce(F.col("borough"), F.lit(BARRIO_DESCONOCIDO)).alias("borough"),
        _franja_del_dia(F.hour("pickup_datetime")).alias("franja"),
        F.col("distancia_mi"),
        F.col("total_pagado"),
    )
    return (
        con_franja.groupBy("fecha", "borough", "franja")
        .agg(
            F.count(F.lit(1)).alias("viajes"),
            F.sum("distancia_mi").alias("distancia"),
            F.sum("total_pagado").alias("ingreso"),
        )
        .select(
            "fecha",
            "borough",
            "franja",
            "viajes",
            F.round("distancia", 2).alias("distancia_total_mi"),
            F.round("ingreso", 2).alias("ingreso_total"),
            F.round(F.col("ingreso") / F.col("distancia"), 2).alias("tarifa_por_milla"),
        )
        .select(*COLUMNAS_TARIFA_POR_MILLA)
    )


# nombre de la tabla en gold -> función que la arma a partir del enriquecido.
TABLAS_GOLD = {
    "tarifa_por_milla_barrio": tarifa_por_milla_por_barrio,
}

# Qué columnas identifican una fila de cada tabla (gold exige que no se repitan ni sean nulas).
LLAVES_DE_GOLD = {
    "tarifa_por_milla_barrio": ("fecha", "borough", "franja"),
}


mi_gold = tarifa_por_milla_por_barrio(enriquecido)
print("Filas de gold:", mi_gold.count(), "| viajes que suma:", mi_gold.agg(F.sum("viajes")).first()[0])
mi_gold.orderBy(F.desc("tarifa_por_milla")).show(8, truncate=False)

**Qué observar.** Cada fila de gold resume muchos viajes. La suma de la columna `viajes` debe ser igual a las filas del enriquecido: gold agrega, no pierde ni inventa viajes. Esa igualdad es la primera regla de conciliación entre silver y gold.

### 8.4 Nuestra función contra la de la casa

La de la casa vive en `gold.py`. Si la nuestra está bien, las dos tablas son idénticas, fila por fila.

In [ ]:
# Celda 31
from pipeline import gold

de_la_casa = gold.tarifa_por_milla_por_barrio(enriquecido)
print("Mismas columnas:", mi_gold.columns == de_la_casa.columns)
print("Mismas filas   :", sorted(map(tuple, mi_gold.collect())) == sorted(map(tuple, de_la_casa.collect())))

Las dos comparaciones deben decir `True`.

## 9. Snowflake: el CREATE TABLE a partir de los tipos de gold

**El problema.** Gold vive en S3 como archivos Parquet. Para consultarla con SQL en Snowflake hay que crear una tabla con **las mismas columnas y tipos compatibles**, y cargarla. La tabla la creas tú, a mano, en Snowsight: el código nunca crea tablas. Si las columnas no coinciden, la carga se detiene antes de tocar nada y dice qué falta; antes, una columna nueva simplemente desaparecía sin aviso.

### 9.1 De los tipos de Spark a los tipos de Snowflake

Spark dice qué tipo tiene cada columna de gold (`dtypes`). Snowflake usa otros nombres para lo mismo:

| Spark | Snowflake |
|---|---|
| `date` | `DATE` |
| `string` | `VARCHAR` |
| `bigint` | `NUMBER(38,0)` |
| `double` | `FLOAT` |

In [ ]:
# Celda 32
print(mi_gold.dtypes)

EQUIVALENCIA = {"date": "DATE", "string": "VARCHAR", "bigint": "NUMBER(38,0)", "double": "FLOAT"}
columnas = ", ".join(f"{nombre.upper()} {EQUIVALENCIA[tipo]}" for nombre, tipo in mi_gold.dtypes)
mi_ddl = f"CREATE TABLE IF NOT EXISTS {{tabla}} ({columnas})"
print()
print(mi_ddl)

### 9.2 Nuestro `CREATE TABLE` contra el de la casa

La casa lo guarda en `carga.py` como `DDL_TARIFA_POR_MILLA`, junto con la lista `TABLAS_A_CARGAR` que dice qué tabla de gold va con qué `CREATE TABLE` (para cargar la tuya, agregas una entrada). Los escribimos aquí igual que allá y comparamos el nuestro, sin importar los espacios:

In [ ]:
# Celda 33
DDL_TARIFA_POR_MILLA = (
    "CREATE TABLE IF NOT EXISTS {tabla} ("
    "FECHA DATE, "
    "BOROUGH VARCHAR, "
    "FRANJA VARCHAR, "
    "VIAJES NUMBER(38,0), "
    "DISTANCIA_TOTAL_MI FLOAT, "
    "INGRESO_TOTAL FLOAT, "
    "TARIFA_POR_MILLA FLOAT)"
)

# Cada entrada: la tabla de gold y el CREATE TABLE que se muestra para su tabla en Snowflake.
TABLAS_A_CARGAR = [
    {"gold": "tarifa_por_milla_barrio", "ddl": DDL_TARIFA_POR_MILLA},
]


def sin_espacios(texto):
    return "".join(texto.split())


print("¿Mi CREATE TABLE es el de la casa?", sin_espacios(mi_ddl) == sin_espacios(DDL_TARIFA_POR_MILLA))


### 9.3 El CREATE TABLE que ejecutas en Snowsight

El nombre de la tabla sale de una función: `nombre_tabla_snowflake(entorno, tabla_de_gold)`. En dev es `BIGDATA.DEV.<TABLA EN MAYÚSCULAS>`. **El código no crea la tabla**: esta celda solo imprime el SQL exacto, con el rol, el warehouse y la base. Cópialo, **ejecútalo en Snowsight** (una hoja de SQL nueva, todo de una vez) y vuelve. Con `IF NOT EXISTS`, repetirlo no hace daño.

In [ ]:
# Celda 34
from pipeline.carga import sql_para_snowsight
from pipeline.herramientas.rutas import nombre_tabla_snowflake

tabla = nombre_tabla_snowflake("dev", "tarifa_por_milla_barrio")
print("Ejecútalo en Snowsight:\n")
print(sql_para_snowsight("dev", tabla, mi_ddl))

Con la tabla creada en Snowsight, en `BIGDATA.DEV`, vacía (todavía no hemos escrito gold en S3), el paso `snowflake` hace dos cosas, en este orden: `verificar_esquema_con_athena` (compara las columnas de gold, tal como las ve Athena, con las de la tabla) y `cargar_gold` (reemplaza ese mes en la tabla con los Parquet de gold). Si la tabla no existe, `cargar_gold` se detiene con un error que imprime este mismo SQL y dice «luego vuelve a correr `cargar_snowflake`». Esas dos las veremos correr cuando ejecutemos el paso con las tareas del DAG.


## 10. Del notebook al `.py`: qué cambiaste y si las pruebas pasan

**El problema.** Un pipeline no corre desde un notebook: corre desde los archivos `.py` que llama el DAG, `make` o un job de Glue. Aquí construiste y probaste cada función, y las que tú cambies tienen que **viajar** a su archivo. Las firmas son las mismas que tienen en los `.py`, así que pasar una función es copiarla completa, sin tocar sus parámetros.

Primero comparamos cada función del notebook con la que trae el repositorio. La celda mira el **código** (ignora los comentarios y las explicaciones entre comillas triples) y, para el `CREATE TABLE`, compara el texto. Si dice `igual`, el `.py` ya tiene esa función y no hay nada que pasar. Si dice `DISTINTO`, **la cambiaste tú en el notebook**: esa es la que tienes que copiar al `.py`. Hoy, antes de que cambies nada, todas deben decir `igual`.

| Función | Va en |
|---|---|
| `agregar_metadatos`, `debe_ingestar`, `registrar_ingesta` | `src/pipeline/bronze.py` |
| `deduplicar` | `src/pipeline/silver.py` |
| `_franja_del_dia`, `tarifa_por_milla_por_barrio` (y `TABLAS_GOLD`, `LLAVES_DE_GOLD`) | `src/pipeline/gold.py` |
| `DDL_TARIFA_POR_MILLA` y `TABLAS_A_CARGAR` | `src/pipeline/carga.py` |

In [ ]:
# Celda 35
import ast
import inspect

from pipeline import bronze, carga, gold, silver


def codigo_sin_docstring(funcion):
    """El código de la función como árbol, sin su docstring: así solo cuenta lo que ejecuta."""
    arbol = ast.parse(inspect.getsource(funcion).lstrip())
    definicion = arbol.body[0]
    cuerpo = definicion.body
    if cuerpo and isinstance(cuerpo[0], ast.Expr) and isinstance(cuerpo[0].value, ast.Constant):
        definicion.body = cuerpo[1:]
    return ast.dump(definicion)


comparaciones = [
    ("agregar_metadatos", agregar_metadatos, bronze.agregar_metadatos, "bronze.py"),
    ("debe_ingestar", debe_ingestar, bronze.debe_ingestar, "bronze.py"),
    ("registrar_ingesta", registrar_ingesta, bronze.registrar_ingesta, "bronze.py"),
    ("deduplicar", deduplicar, silver.deduplicar, "silver.py"),
    ("_franja_del_dia", _franja_del_dia, gold._franja_del_dia, "gold.py"),
    ("tarifa_por_milla_por_barrio", tarifa_por_milla_por_barrio, gold.tarifa_por_milla_por_barrio, "gold.py"),
]
for nombre, del_notebook, del_modulo, archivo in comparaciones:
    firma_igual = list(inspect.signature(del_notebook).parameters) == list(
        inspect.signature(del_modulo).parameters
    )
    codigo_igual = codigo_sin_docstring(del_notebook) == codigo_sin_docstring(del_modulo)
    print(f"{nombre:<30}{archivo:<11}firma: {'igual' if firma_igual else 'DISTINTA':<10}código: {'igual' if codigo_igual else 'DISTINTO'}")
print(f"{'DDL_TARIFA_POR_MILLA':<30}{'carga.py':<11}texto: {'igual' if DDL_TARIFA_POR_MILLA == carga.DDL_TARIFA_POR_MILLA else 'DISTINTO'}")


Ahora las pruebas del equipo. `probar_mis_funciones` recibe las funciones **de este notebook**, las pone un instante en los módulos del pipeline, corre `tests/test_equipo_bronze.py` y `tests/test_equipo_silver.py` (las mismas de `make test`), imprime una línea por prueba y deja todo como estaba. Cada prueba sale `OK`, `PENDIENTE` o `FALLA`. Una sale `PENDIENTE` mientras la función conserve la marca `# PENDIENTE(equipo)`; cuando escribes la función y borras la marca, esa misma prueba corre de verdad.

Antes de cambiar nada debes ver `OK` en todo lo que toca la puerta de la tabla de control y `PENDIENTE` en lo de los metadatos y en lo de deduplicar: justo los dos problemas que dijimos en voz alta. Cuando tu equipo escriba esas dos funciones aquí y vuelva a correr esta celda, esas líneas pasan a `OK`, y entonces sí se copian al `.py`.

In [ ]:
# Celda 36
from pipeline.herramientas.equipo import probar_mis_funciones

codigo = probar_mis_funciones(
    agregar_metadatos=agregar_metadatos,
    debe_ingestar=debe_ingestar,
    registrar_ingesta=registrar_ingesta,
    deduplicar=deduplicar,
)
print("Código de salida de pytest:", codigo, "(0 = ninguna prueba falló)")


## 11. Cierre: apagar Spark

Hemos probado la lógica de las cuatro partes del pipeline: **bronze** (con su puerta y su registro), **silver** (con el problema de los dos lotes), **gold** (con una pregunta de la casa) y **snowflake** (con su `CREATE TABLE`), y comprobamos qué funciones del notebook difieren de los `.py`.

Apagamos Spark para liberar la memoria: en esta máquina solo cabe una sesión a la vez, y los comandos de `make` abren la suya.

> **Qué dejó este notebook en dev.** El archivo crudo y el lote de bronze en `s3://<tu-lake>/dev/`, las filas de la tabla de control (las de enero y las de los meses de ensayo de 2023) y la tabla vacía de gold en `BIGDATA.DEV`. Antes de pasar las funciones a los `.py` corremos `resetear_entorno` en dev para empezar limpios: así las tareas del DAG parten de un dev sin lotes ni registros de este ensayo.

In [ ]:
# Celda 37
spark.stop()
print("Spark apagado.")